# SoFIFA cleaning (revised, validated, deduped, feature‑engineered)

This notebook replaces the earlier *SoFIFA_cleaning.ipynb* with a more robust pipeline:

- **Validation**: schema checks, corruption checks (e.g., URLs appearing in numeric columns)
- **Repair** (optional but included): line-by-line CSV realignment using the `url` anchor if the source CSV is comma-corrupted
- **Deduplication**: one row per `player_id` (keeps the latest `version`)
- **Feature engineering**: age, contract years left (capped), position flags, log transforms, optional composite ratings
- **Output**: **Parquet** (primary) + a small CSV preview for quick inspection

> **Inputs** (expected in `/mnt/data/`):  
> - `player_stats.csv` (raw SoFIFA export/scrape)

> **Outputs** (written to `/mnt/data/`):  
> - `player_stats_cleaned.parquet`  
> - `player_stats_cleaned.csv` (optional preview/sample)


In [1]:
# --- Imports ---
import os
import re
import math
import pandas as pd
import numpy as np
from datetime import date

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


## 1) Configuration

In [2]:
# --- Paths ---
RAW_CSV_PATH = '../player_stats.csv'

# Primary output (recommended for modelling)
OUT_PARQUET_PATH = "../player_stats_cleaned.parquet"

# Optional CSV export (handy for quick viewing / sharing; will not preserve dtypes as well as Parquet)
OUT_CSV_PATH = "../player_stats_cleaned.csv"


# --- Business rules ---
CONTRACT_YEARS_CAP = 10

# --- Repair settings ---
# If your raw CSV is well-formed, you can set this to False and it will do a normal pd.read_csv.
USE_REPAIR_READER = True


## 2) Robust CSV reader with optional repair

### Why this exists
SoFIFA scrapes often contain comma-separated lists (e.g., play styles, specialties).  
If those fields are not properly quoted at export time, plain CSV parsing will **shift columns**.

This reader uses a practical anchor:
- **the SoFIFA `url` token** (e.g., `https://sofifa.com/player/...`)  
and forces that token into the `url` column position by inserting/merging tokens as needed.

If your raw file is already clean, you may turn `USE_REPAIR_READER = False` in config.


In [3]:
def read_csv_with_url_anchor_repair(
    csv_path: str,
    encoding: str = "utf-8",
    expected_url_col: str = "url",
    url_prefix: str = "https://sofifa.com/player/",
) -> pd.DataFrame:
    """Read a CSV that may be column-shift corrupted by unquoted commas in text fields.

    Strategy:
    - Read header to get expected column names and count.
    - For each line:
        - Split by comma (naive).
        - Find the token that looks like a SoFIFA player URL.
        - Force that token to land in the 'url' column index by inserting blanks (if URL is early)
          or merging surplus tokens (if URL is late).
        - Pad/truncate to the expected number of columns.
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"File not found: {csv_path}")

    with open(csv_path, "r", encoding=encoding, errors="replace") as f:
        header_line = f.readline().rstrip("\n")
        columns = header_line.split(",")
        ncols = len(columns)

        if expected_url_col not in columns:
            raise ValueError(
                f"Expected column '{expected_url_col}' not found in header. "
                f"Columns found: {columns[:10]} ... ({ncols} total)"
            )

        url_idx_expected = columns.index(expected_url_col)

        rows = []
        bad_lines = 0
        no_url_lines = 0

        for line_num, line in enumerate(f, start=2):
            line = line.rstrip("\n")
            # Skip completely empty lines
            if not line.strip():
                continue

            tokens = line.split(",")

            # Find a token that looks like a SoFIFA player URL
            url_positions = [i for i, tok in enumerate(tokens) if tok.startswith(url_prefix)]

            if not url_positions:
                # If there is no URL token, we can still attempt to pad/truncate.
                no_url_lines += 1
                if len(tokens) < ncols:
                    tokens = tokens + [""] * (ncols - len(tokens))
                elif len(tokens) > ncols:
                    # Merge overflow into the last column
                    tokens = tokens[:ncols-1] + [",".join(tokens[ncols-1:])]
                rows.append(tokens)
                continue

            # If multiple URL-like tokens exist, keep the last (most likely the true url field)
            url_idx_found = url_positions[-1]

            # If URL is too early: insert blanks right before it until it reaches expected index
            if url_idx_found < url_idx_expected:
                insert_count = url_idx_expected - url_idx_found
                tokens = tokens[:url_idx_found] + [""] * insert_count + tokens[url_idx_found:]
                url_idx_found = url_idx_expected

            # If URL is too late: merge the surplus tokens into the column immediately before expected URL
            # This typically happens when a text field contains extra commas and wasn't quoted.
            if url_idx_found > url_idx_expected:
                # tokens[url_idx_expected:url_idx_found] are extra pieces that pushed URL right
                # Merge them into the previous field (url_idx_expected - 1), because that's usually
                # the last text column before the URL.
                if url_idx_expected - 1 >= 0:
                    surplus = tokens[url_idx_expected:url_idx_found]
                    tokens[url_idx_expected - 1] = (
                        (tokens[url_idx_expected - 1] or "") + ("," if tokens[url_idx_expected - 1] else "") + ",".join(surplus)
                    )
                    # Remove the surplus slice
                    del tokens[url_idx_expected:url_idx_found]
                # Recompute found index
                url_idx_found = tokens.index(next(tok for tok in tokens if tok.startswith(url_prefix)))

            # Final length correction
            if len(tokens) < ncols:
                tokens = tokens + [""] * (ncols - len(tokens))
            elif len(tokens) > ncols:
                # Merge overflow into the last column
                tokens = tokens[:ncols-1] + [",".join(tokens[ncols-1:])]

            if len(tokens) != ncols:
                bad_lines += 1

            rows.append(tokens)

    df = pd.DataFrame(rows, columns=columns)

    print(f"Loaded rows: {len(df):,}")
    print(f"Header columns: {ncols}")
    print(f"Lines without URL anchor: {no_url_lines:,}")
    if bad_lines:
        print(f"WARNING: {bad_lines:,} lines still had unexpected column counts after repair.")

    return df


def read_raw_sofifa(csv_path: str, use_repair_reader: bool = True) -> pd.DataFrame:
    if use_repair_reader:
        return read_csv_with_url_anchor_repair(csv_path)
    # Normal read (works only if the CSV is well-formed / properly quoted)
    return pd.read_csv(csv_path, low_memory=False)


## 3) Load + first-pass validation

In [4]:
df_raw = read_raw_sofifa(RAW_CSV_PATH, use_repair_reader=USE_REPAIR_READER)

print(df_raw.shape)
display(df_raw.head(3))


Loaded rows: 20,586
Header columns: 76
Lines without URL anchor: 2
(20586, 76)


,player_id,version,name,full_name,description,image,height_cm,weight_kg,dob,positions,overall_rating,potential,value,wage,preferred_foot,weak_foot,skill_moves,international_reputation,body_type,real_face,release_clause,specialities,club_id,club_name,club_league_id,club_league_name,club_logo,club_rating,club_position,club_kit_number,club_joined,club_contract_valid_until,country_id,country_name,country_league_id,country_league_name,country_flag,country_rating,country_position,country_kit_number,attacking_crossing,attacking_finishing,attacking_heading_accuracy,attacking_short_passing,attacking_volleys,skill_dribbling,skill_curve,skill_fk_accuracy,skill_long_passing,skill_ball_control,movement_acceleration,movement_sprint_speed,movement_agility,movement_reactions,movement_balance,power_shot_power,power_jumping,power_stamina,power_strength,power_long_shots,mentality_aggression,mentality_interceptions,mentality_vision,mentality_penalties,mentality_composure,defending_defensive_awareness,defending_standing_tackle,defending_sliding_tackle,goalkeeping_gk_diving,goalkeeping_gk_handling,goalkeeping_gk_kicking,goalkeeping_gk_positioning,goalkeeping_gk_reflexes,play_styles,url,mentality_attack_position
0,239085,FC 26,E. Haaland,Erling Haaland,"""Erling Haaland (Erling Braut Håland",born 21 July 2000) is a Norwegian footballer ...,and the Norway national team. In the game FC 26,"his overall rating is 91.""",https://cdn.sofifa.net/players/239/085/26_360.png,195,94,2000-07-21,ST,91,93,172500000,390000,Left,3,3,5,Unique,Yes,332100000,"""#Aerial threat",#Distance shooter,#Strength,#Clinical finisher,"#Complete forward""",10,Manchester City,13,Premier League,https://cdn.sofifa.net/meta/team/9/30.png,5,ST,9,"""Jul 1","2022""",2034,1352,Norway,78,Friendly International,https://cdn.sofifa.net/flags/undefined.png,5,LS,9,58,96,85,78,90,79,77,62,66,83,82,92,71,94,69,94,93,78,93,83,88,43,75,90,86,"42,47,29,7,14,13,11,7,""Low driven shot, Chip s...",https://sofifa.com/player/239085/erling-haalan...,95
1,231747,FC 26,K. Mbappé,Kylian Mbappé,"""Kylian Mbappé (Kylian Mbappé Lottin",born 20 December 1998) is a French footballer...,and the France national team. In the game FC 26,"his overall rating is 91.""",https://cdn.sofifa.net/players/231/747/26_360.png,182,81,1998-12-20,ST,91,92,157000000,610000,Right,4,5,5,Unique,Yes,333600000,"""#Speedster",#Dribbler,#Distance shooter,#Acrobat,#Clinical finisher,"#Complete forward""",243,Real Madrid,53,La Liga,https://cdn.sofifa.net/meta/team/3468/30.png,5,RS,10,"""Jul 1","2024""",2029,1335,France,78,Friendly International,https://cdn.sofifa.net/flags/undefined.png,5,ST,10,78,94,78,87,87,92,80,69,74,93,97,97,93,91,82,91,90,83,77,86,61,38,83,84,"88,26,34,32,13,5,7,11,6,""Quick step, Finesse s...",https://sofifa.com/player/231747/kylian-mbappe...,91
2,255253,FC 26,Vitinha,Vítor Machado Ferreira,"""Vitinha (born 13 February 2000) is a Portugue...",and the Portugal national team. In the game F...,"his overall rating is 90.""",https://cdn.sofifa.net/players/255/253/26_360.png,172,64,2000-02-13,"""CM",CDM,"CAM""",90,92,149000000,185000,Right,3,4,5,Lean (170-185),Yes,286800000,"""#Dribbler",#Playmaker,#Acrobat,"#Complete midfielder""",73,Paris Saint-Germain,16,Ligue 1,https://cdn.sofifa.net/meta/team/591/30.png,5,CM,17,"""Jun 30","2022""",2029,1354,Portugal,78,Friendly International,https://cdn.sofifa.net/flags/undefined.png,5,CDM,23,78,81,52,91,62,91,83,69,91,90,75,69,92,90,89,78,66,89,59,89,77,85,91,88,88,"74,79,69,12,13,8,5,5,""Technical, Finesse shot,...",https://sofifa.com/player/255253/vitor-machado...,81


In [5]:
# --- Basic schema checks ---
required_cols = ["player_id", "version", "name", "positions", "dob", "url"]
missing = [c for c in required_cols if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# --- Corruption checks (URLs in numeric columns is a common symptom of column shifting) ---
# We'll test a few numeric columns if present.
numeric_sentinel_cols = [c for c in ["overall_rating", "potential", "defending_standing_tackle", "goalkeeping_gk_diving"] if c in df_raw.columns]

def frac_looks_like_url(s: pd.Series) -> float:
    s = s.astype(str)
    return s.str.contains("sofifa.com/player", na=False).mean()

for c in numeric_sentinel_cols:
    frac = frac_looks_like_url(df_raw[c])
    print(f"{c}: {frac:.3%} of values look like URLs")

# If any sentinel column has a high URL fraction, warn loudly.
high_url_cols = [c for c in numeric_sentinel_cols if frac_looks_like_url(df_raw[c]) > 0.01]
if high_url_cols:
    print("\nWARNING: Possible misalignment detected (URL-like strings in numeric columns):", high_url_cols)
    print("If USE_REPAIR_READER=False, set it to True and reload. If it is already True, consider re-exporting the CSV with proper quoting.")


overall_rating: 0.000% of values look like URLs
potential: 0.000% of values look like URLs
defending_standing_tackle: 0.000% of values look like URLs
goalkeeping_gk_diving: 0.000% of values look like URLs


## 4) Type cleaning + engineered features

In [ ]:
df = df_raw.copy()

# --- Standardize empty strings to NaN ---
df = df.replace({"": np.nan, "None": np.nan, "nan": np.nan})

# --- Convert obvious numeric columns ---
# We keep this broad but safe: any column that is mostly numeric should become numeric.
def coerce_numeric_if_mostly_numeric(df: pd.DataFrame, threshold: float = 0.85) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if out[col].dtype == "object":
            s = out[col]
            # Quick skip for known textual fields
            if col in {"name", "full_name", "description", "image", "positions", "play_styles", "specialities", "url", "country_name", "club_name", "league_name"}:
                continue

            # Estimate numeric-ness on a sample
            sample = s.dropna().astype(str).head(2000)
            if sample.empty:
                continue
            numeric_like = sample.str.match(r"^-?\d+(\.\d+)?$").mean()

            if numeric_like >= threshold:
                out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

df = coerce_numeric_if_mostly_numeric(df)

# --- Parse dates ---
df["dob"] = pd.to_datetime(df["dob"], errors="coerce")

if "club_contract_valid_until" in df.columns:
    # Some datasets store this as a year, some as a date; try both.
    # 1) numeric year
    tmp_year = pd.to_numeric(df["club_contract_valid_until"], errors="coerce")
    # 2) datetime fallback
    tmp_date = pd.to_datetime(df["club_contract_valid_until"], errors="coerce")
    df["club_contract_valid_until_year"] = tmp_year
    df.loc[df["club_contract_valid_until_year"].isna() & tmp_date.notna(), "club_contract_valid_until_year"] = tmp_date.dt.year
else:
    df["club_contract_valid_until_year"] = np.nan

# --- Version → reference year (heuristic) ---
# SoFIFA 'version' often starts with the 2-digit year (e.g., 2600xx for 2026).
# If your version scheme differs, adjust this.
df["version"] = pd.to_numeric(df["version"], errors="coerce")

def infer_version_year(v):
    if pd.isna(v):
        return np.nan
    v = int(v)
    yy = int(str(v)[:2])  # first two digits
    # Map 00-79 -> 2000-2079, 80-99 -> 1980-1999 (rare; included for completeness)
    return 2000 + yy if yy <= 79 else 1900 + yy

df["version_year"] = df["version"].apply(infer_version_year)

# --- Age at version year (as of July 1 of version year) ---
def age_on_july1(dob, year):
    if pd.isna(dob) or pd.isna(year):
        return np.nan
    ref = pd.Timestamp(year=int(year), month=7, day=1)
    return (ref - dob).days / 365.25

df["age_at_version"] = [age_on_july1(d, y) for d, y in zip(df["dob"], df["version_year"])]

# --- Contract years left (capped) ---
# years_left = contract_valid_until_year - version_year
df["contract_years_left"] = df["club_contract_valid_until_year"] - df["version_year"]
df.loc[df["contract_years_left"] < 0, "contract_years_left"] = 0
df["contract_years_left"] = df["contract_years_left"].clip(upper=CONTRACT_YEARS_CAP)

df["contract_unknown"] = df["club_contract_valid_until_year"].isna().astype(int)

# --- Positions: primary + multi-hot flags ---
def split_positions(x):
    if pd.isna(x):
        return []
    return [p.strip() for p in str(x).split(",") if p.strip()]

df["positions_list"] = df["positions"].apply(split_positions)
df["primary_position"] = df["positions_list"].apply(lambda lst: lst[0] if len(lst) else np.nan)

ALL_POS = ["GK","CB","LB","RB","LWB","RWB","CDM","CM","CAM","LM","RM","LW","RW","CF","ST"]
for p in ALL_POS:
    df[f"is_{p}"] = df["positions_list"].apply(lambda lst, p=p: int(p in lst))

# --- Log transforms for monetary columns (if present) ---
for money_col in ["value", "wage", "release_clause"]:
    if money_col in df.columns:
        df[money_col] = pd.to_numeric(df[money_col], errors="coerce")
        df[f"log1p_{money_col}"] = np.log1p(df[money_col])

# --- Optional: compact skill aggregates (only if the underlying columns exist) ---
def mean_of_existing(cols):
    existing = [c for c in cols if c in df.columns]
    if not existing:
        return None
    return pd.to_numeric(df[existing], errors="coerce").mean(axis=1)

# Basic examples (adjust to taste)
df["agg_pace"] = mean_of_existing(["movement_acceleration", "movement_sprint_speed"])
df["agg_shooting"] = mean_of_existing(["attacking_finishing", "power_shot_power", "skill_fk_accuracy", "attacking_volleys", "mentality_penalties"])
df["agg_passing"] = mean_of_existing(["attacking_short_passing", "skill_long_passing", "skill_curve", "skill_crossing", "skill_vision"])
df["agg_dribbling"] = mean_of_existing(["skill_dribbling", "skill_ball_control", "movement_agility", "movement_balance", "movement_reactions"])
df["agg_defending"] = mean_of_existing(["mentality_interceptions", "defending_marking_awareness", "defending_standing_tackle", "defending_sliding_tackle"])
df["agg_physical"] = mean_of_existing(["power_strength", "power_stamina", "power_jumping"])

print("Post-cleaning shape:", df.shape)
display(df.head(3))


C:\Users\Praet\AppData\Local\Temp\ipykernel_11200\1729855760.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["dob"] = pd.to_datetime(df["dob"], errors="coerce")
C:\Users\Praet\AppData\Local\Temp\ipykernel_11200\1729855760.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tmp_date = pd.to_datetime(df["club_contract_valid_until"], errors="coerce")


TypeError: dtype 'str' does not support operation 'mean'

## 5) Deduplication (keep latest version per player_id)

Rule:
- group by `player_id`
- keep the row with the **maximum `version`**  
If `version` is missing, it falls back to the first occurrence.

You can change the rule (e.g., highest overall_rating) if it better fits your use case.


In [ ]:
# Ensure player_id is numeric (but keep as Int64 to allow missing safely)
df["player_id"] = pd.to_numeric(df["player_id"], errors="coerce").astype("Int64")

# Sort so that latest version comes first within each player_id
df_sorted = df.sort_values(by=["player_id", "version"], ascending=[True, False])

before = len(df_sorted)
df_dedup = df_sorted.drop_duplicates(subset=["player_id"], keep="first").copy()
after = len(df_dedup)

print(f"Rows before dedupe: {before:,}")
print(f"Rows after dedupe : {after:,}")
print(f"Removed          : {before-after:,} duplicates")


## 6) Final validation report

In [ ]:
def missingness_report(df: pd.DataFrame, top_n: int = 25) -> pd.DataFrame:
    miss = df.isna().mean().sort_values(ascending=False)
    return miss.head(top_n).to_frame("missing_fraction")

report = missingness_report(df_dedup, top_n=30)
display(report)

# Quick sanity checks
assert df_dedup["player_id"].isna().mean() < 0.01, "Too many missing player_id values; something is wrong upstream."

# URL should look like a URL most of the time
url_ok = df_dedup["url"].astype(str).str.startswith("https://sofifa.com/player/").mean()
print(f"URL field looks valid in {url_ok:.2%} of rows")


## 7) Output to Parquet (primary) + optional CSV preview

In [ ]:
# Parquet preserves dtypes and is preferred for modelling
df_dedup.to_parquet(OUT_PARQUET_PATH, index=False)
print("Wrote:", OUT_PARQUET_PATH)

# Optional CSV preview (handy if you want a quick glance outside Python)
if CSV_PREVIEW_N is None:
    df_dedup.to_csv(OUT_CSV_PATH, index=False)
    print("Wrote:", OUT_CSV_PATH)
else:
    df_dedup.head(CSV_PREVIEW_N).to_csv(OUT_CSV_PATH, index=False)
    print(f"Wrote preview ({CSV_PREVIEW_N:,} rows):", OUT_CSV_PATH)
